In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f  # Spark's built-in functions (like SQL functions)
from pyspark.sql.types import (
    StructType,      # For defining complex schemas
    StructField,     # Individual field in a schema
    StringType,      # String data type
    IntegerType,     # Integer data type
    DoubleType,      # Floating point data type
    TimestampType,   # Timestamp data type
)

# Create SparkSession - the entry point for all Spark operations
# The builder pattern lets us configure the session before creating it
spark = SparkSession.builder \
    .appName("Spark40_Imperative_Pipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Reduce log verbosity - we only want warnings and errors, not info messages
# This keeps our notebook output clean
spark.sparkContext.setLogLevel("WARN")

# Print version to confirm we're using Spark
print(f"Spark Version: {spark.version}")
print(f"Application Name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print("\n[OK] SparkSession created successfully")

Spark Version: 3.5.0
Application Name: Spark40_Imperative_Pipeline
Master: local[*]

[OK] SparkSession created successfully


In [2]:
import os
DATA_PATH = "/home/jovyan/data" if os.path.exists("/home/jovyan/data") else "/data"

def load_dimension_table(name: str, path: str):
    """
    Load a dimension table from parquet and register as temp view.
    
    This is a REUSABLE helper function - a common pattern in imperative code.
    We define generic functions and call them with different parameters.
    
    Args:
        name: Target table name (e.g., "dim_brands")
        path: Source parquet file path
    
    Returns:
        The loaded DataFrame (also registered as temp view)
    """
    # Read parquet file - Spark automatically infers the schema
    df = spark.read.parquet(path)
    
    # Register as temp view (in production, would write to Iceberg)
    # Temp views are session-scoped and perfect for learning
    df.createOrReplaceTempView(f"bronze_{name}")
    
    # Return the DataFrame - this allows direct inspection!
    return df

# Define our dimension tables - this is our "configuration"
# In imperative code, we often separate config from logic
DIMENSION_TABLES = {
    "dim_categories": f"{DATA_PATH}/dimensions/categories.parquet",
    "dim_brands": f"{DATA_PATH}/dimensions/brands.parquet",
    "dim_items": f"{DATA_PATH}/dimensions/items.parquet",
    "dim_locations": f"{DATA_PATH}/dimensions/locations.parquet",
}

# Execute: Load each dimension table
# We MANUALLY iterate and call our function for each table
print("Loading dimension tables to Bronze layer...")
print("=" * 50)

# Store returned DataFrames for verification
loaded_dfs = {}
for table_name, file_path in DIMENSION_TABLES.items():
    df = load_dimension_table(table_name, file_path)
    loaded_dfs[table_name] = df
    print(f"  [OK] bronze_{table_name}: {df.count():,} rows")

print("\n[OK] All dimension tables loaded")

# =============================================================================
# VERIFICATION: Functions return DataFrames that you can inspect directly!
# =============================================================================
print("\n" + "=" * 50)
print("VERIFY: load_dimension_table() returns DataFrame")
print("=" * 50)

brands_df = loaded_dfs["dim_brands"]
print(f"\nType: {type(brands_df).__name__}")
print(f"Count: {brands_df.count()}")
brands_df.show(5, truncate=False)

Loading dimension tables to Bronze layer...
  [OK] bronze_dim_categories: 10 rows
  [OK] bronze_dim_brands: 20 rows
  [OK] bronze_dim_items: 160 rows
  [OK] bronze_dim_locations: 4 rows

[OK] All dimension tables loaded

VERIFY: load_dimension_table() returns DataFrame

Type: DataFrame
Count: 20
+---+---------------+------------+------------------+--------+
|id |name           |cuisine_type|avg_prep_time_mins|momentum|
+---+---------------+------------+------------------+--------+
|1  |Burger Republic|American    |12                |stable  |
|2  |Wok This Way   |Chinese     |15                |growing |
|3  |Pizza Planet   |Italian     |18                |stable  |
|4  |Taco Tornado   |Mexican     |10                |growing |
|5  |Sushi Express  |Japanese    |20                |growing |
+---+---------------+------------+------------------+--------+
only showing top 5 rows



In [3]:
@pipeline.materialized_view(name="bronze_dim_categories", layer="bronze")
def dim_categories():
    """Food categories dimension table."""
    return spark.read.parquet(f"{DATA_PATH}/dimensions/categories.parquet")


@pipeline.materialized_view(name="bronze_dim_brands", layer="bronze")
def dim_brands():
    """Ghost kitchen brands dimension table."""
    return spark.read.parquet(f"{DATA_PATH}/dimensions/brands.parquet")


@pipeline.materialized_view(name="bronze_dim_items", layer="bronze")
def dim_items():
    """Menu items dimension table."""
    return spark.read.parquet(f"{DATA_PATH}/dimensions/items.parquet")


@pipeline.materialized_view(name="bronze_dim_locations", layer="bronze")
def dim_locations():
    """Delivery locations dimension table."""
    return spark.read.parquet(f"{DATA_PATH}/dimensions/locations.parquet")


print("[OK] Dimension tables DEFINED")
print(f"  Registered tables: {list(pipeline.tables.keys())}")
print("\n" + "=" * 50)
print("VERIFY: Call a decorated function directly to see its output:")
print("=" * 50)

# You can call the decorated function directly - it returns the DataFrame!
brands_df = dim_brands()
print(f"\ndim_brands() returns: {type(brands_df)}")
print(f"Row count: {brands_df.count()}")
brands_df.show(5, truncate=False)

[OK] Dimension tables DEFINED
  Registered tables: ['bronze_dim_categories', 'bronze_dim_brands', 'bronze_dim_items', 'bronze_dim_locations']

VERIFY: Call a decorated function directly to see its output:


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/data/dimensions/brands.parquet.